# 03 Training Science — Optimisers

**Status:** Complete

**Learning outcome:** Implement SGD, Momentum, RMSProp, Adam, and AdamW from scratch, then race them on the Beale function from (-3,-3) to visualise convergence behaviour and understand why AdamW is the default choice.

## Why This Matters

The optimiser determines how gradient information is used to update weights. SGD can oscillate and get stuck; momentum smooths the path; Adam adapts per-parameter learning rates. Understanding these differences is essential for tuning training — the wrong optimiser or hyperparameters can make the difference between convergence and failure.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Beale Function

$f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$

Global minimum at $(3, 0.5)$ with $f(3, 0.5) = 0$. We start all optimisers at $(-3, -3)$ — far from the optimum.

In [2]:
def beale(x, y):
    return ((1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2
            + (2.625 - x + x*y**3)**2)

def beale_grad(x, y):
    t1, t2, t3 = 1.5 - x + x*y, 2.25 - x + x*y**2, 2.625 - x + x*y**3
    dx = 2*t1*(-1+y) + 2*t2*(-1+y**2) + 2*t3*(-1+y**3)
    dy = 2*t1*(x) + 2*t2*(2*x*y) + 2*t3*(3*x*y**2)
    return np.array([dx, dy])

# Verify gradient at optimum
assert beale(3, 0.5) < 1e-10, "Beale minimum wrong"
g = beale_grad(3, 0.5)
assert np.linalg.norm(g) < 1e-8, "Gradient at optimum not zero"
print(f"Beale(3, 0.5) = {beale(3, 0.5):.2e}, ||grad|| = {np.linalg.norm(g):.2e} ✓")

Beale(3, 0.5) = 0.00e+00, ||grad|| = 0.00e+00 ✓


## Optimiser Implementations (From Scratch)

In [3]:
def run_optimiser(name, lr, steps, x0=np.array([-3.0, -3.0])):
    """Run an optimiser on Beale and return trajectory."""
    w = x0.copy()
    path = [w.copy()]

    # State variables
    m = np.zeros(2)  # first moment (momentum / Adam)
    v = np.zeros(2)  # second moment (RMSProp / Adam)
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    wd = 0.01  # weight decay for AdamW

    for t in range(1, steps + 1):
        g = beale_grad(w[0], w[1])
        # Clip gradients to prevent divergence
        g = np.clip(g, -100, 100)

        if name == 'SGD':
            w = w - lr * g

        elif name == 'Momentum':
            m = 0.9 * m + g
            w = w - lr * m

        elif name == 'RMSProp':
            v = 0.9 * v + 0.1 * g**2
            w = w - lr * g / (np.sqrt(v) + eps)

        elif name == 'Adam':
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            w = w - lr * m_hat / (np.sqrt(v_hat) + eps)

        elif name == 'AdamW':
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            w = w - lr * (m_hat / (np.sqrt(v_hat) + eps) + wd * w)

        path.append(w.copy())

    return np.array(path)

# Run all optimisers
steps = 5000
configs = {
    'SGD':      {'lr': 1e-5, 'steps': steps},
    'Momentum': {'lr': 1e-5, 'steps': steps},
    'RMSProp':  {'lr': 1e-3, 'steps': steps},
    'Adam':     {'lr': 1e-2, 'steps': steps},
    'AdamW':    {'lr': 1e-2, 'steps': steps},
}

paths = {}
for name, cfg in configs.items():
    paths[name] = run_optimiser(name, **cfg)
    final = paths[name][-1]
    final_loss = beale(final[0], final[1])
    print(f"{name:>10s}: final pos ({final[0]:+.4f}, {final[1]:+.4f}), loss = {final_loss:.4f}")

       SGD: final pos (-0.4340, -0.7178), loss = 21.4615
  Momentum: final pos (+1.8173, -0.0374), loss = 0.9901
   RMSProp: final pos (+1.9723, +0.1596), loss = 0.5689
      Adam: final pos (+2.8832, +0.4693), loss = 0.0025
     AdamW: final pos (+2.7076, +0.4175), loss = 0.0193


---
**Understanding Why SGD Oscillates in Ravines**

**Plain language:** Imagine walking downhill wearing a blindfold — you can only feel the slope right under your feet. If you're in a narrow canyon, the steepest direction points into the canyon wall, not along the canyon floor. So you bounce left-right between the walls instead of walking smoothly toward the exit.

**Building intuition:** SGD uses a single, fixed learning rate $\eta$ for every parameter. The gradient vector $\nabla L$ points in the direction of steepest ascent *locally*, but in regions where curvature differs dramatically across dimensions (high *condition number*), the gradient is dominated by the steep dimension. The flat dimension — which actually leads toward the minimum — barely contributes. The result is zig-zagging across the steep axis with slow net progress along the shallow one.

**Formal statement:** For a quadratic bowl $L(\mathbf{w}) = \tfrac{1}{2}\mathbf{w}^\top H \mathbf{w}$ with Hessian eigenvalues $\lambda_{\max} \gg \lambda_{\min}$, SGD converges at rate $\mathcal{O}\!\bigl(\bigl(\tfrac{\lambda_{\max} - \lambda_{\min}}{\lambda_{\max} + \lambda_{\min}}\bigr)^{2t}\bigr)$. When the condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ is large, the step $\Delta \mathbf{w} = -\eta \nabla L$ oscillates along the eigenvector of $\lambda_{\max}$ while making negligible progress along the eigenvector of $\lambda_{\min}$.

---

---
**Understanding What Momentum Does (Exponential Moving Average)**

**Plain language:** Think of a heavy ball rolling downhill. It doesn't instantly change direction with every bump — it carries speed from before. If the hill consistently slopes one way, the ball accelerates. If the hill keeps reversing, the ball's momentum cancels out the back-and-forth. That's exactly what momentum does for gradient descent: it smooths out the noise and builds up speed in directions that matter.

**Building intuition:** At each step, instead of using only the current gradient, momentum maintains a *velocity* vector that is an exponential moving average (EMA) of past gradients. The decay factor $\beta$ (typically 0.9) controls memory length: $\sim 1/(1-\beta) = 10$ steps. Gradients that consistently point the same way accumulate into a large velocity; oscillating gradients cancel out. This damps the zig-zag problem of vanilla SGD and accelerates convergence along low-curvature directions.

**Formal statement:** The momentum update is $\mathbf{m}_t = \beta \mathbf{m}_{t-1} + \nabla L(\mathbf{w}_{t-1})$, $\;\mathbf{w}_t = \mathbf{w}_{t-1} - \eta \mathbf{m}_t$. Expanding, $\mathbf{m}_t = \sum_{i=0}^{t-1} \beta^{t-1-i} \nabla L(\mathbf{w}_i)$, an exponentially-weighted sum. For a quadratic, momentum improves the convergence rate to $\mathcal{O}\!\bigl(\bigl(\tfrac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\bigr)^{t}\bigr)$, a quadratic improvement over vanilla SGD in the condition number $\kappa$.

---

---
**Understanding Adaptive Learning Rates (RMSProp)**

**Plain language:** Imagine you're adjusting the volume on a mixing board with many channels. Some channels are blasting loud (big gradients), others are barely audible (small gradients). Instead of turning every knob by the same amount, you scale your adjustment by how loud each channel has been recently — big turns for the quiet ones, gentle turns for the loud ones. RMSProp does this for each parameter automatically.

**Building intuition:** RMSProp tracks a running average of squared gradients for each parameter individually. Parameters that have received large gradients recently have a large denominator, so their effective learning rate shrinks. Parameters with consistently small gradients get a larger effective step. This per-parameter adaptation means the optimiser no longer needs a single learning rate to work across dimensions with wildly different curvatures — each dimension is handled independently.

**Formal statement:** RMSProp maintains $v_t = \gamma\, v_{t-1} + (1-\gamma)\, g_t^2$ (element-wise), where $\gamma$ is typically 0.9. The update is $w_t = w_{t-1} - \tfrac{\eta}{\sqrt{v_t} + \epsilon}\, g_t$. The effective per-parameter learning rate is $\eta_{\text{eff},i} = \eta / (\sqrt{v_{t,i}} + \epsilon)$, which approximates $\eta / \text{RMS}(g_{1:t,i})$. This scales inversely with the root-mean-square of recent gradients, equalising progress across dimensions of different curvature.

---

---
**Understanding Adam = Momentum + Adaptive LR + Bias Correction**

**Plain language:** Adam is like combining two good ideas at once. From momentum, it gets a sense of which direction to go (smoothing out noisy gradients). From RMSProp, it gets a sense of how big a step to take for each parameter. The trick is that both of these estimates start at zero, which would make the first few steps far too small — so Adam includes a bias correction that compensates for the cold start.

**Building intuition:** Adam maintains two exponential moving averages: $m_t$ (first moment, like momentum) tracks the mean gradient direction, and $v_t$ (second moment, like RMSProp) tracks the mean squared gradient magnitude. Because both are initialised at zero, the raw estimates are biased toward zero in the early steps — for example, $m_1 = 0.1 \cdot g_1$ instead of $g_1$. The bias-corrected estimates $\hat{m}_t = m_t / (1 - \beta_1^t)$ and $\hat{v}_t = v_t / (1 - \beta_2^t)$ fix this, ensuring the first steps are properly scaled. AdamW further improves on Adam by decoupling weight decay from the adaptive gradient scaling.

**Formal statement:** Adam computes: $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$, $\;v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$, $\;\hat{m}_t = m_t/(1-\beta_1^t)$, $\;\hat{v}_t = v_t/(1-\beta_2^t)$, $\;w_t = w_{t-1} - \eta\, \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$. Defaults: $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$. The bias correction ensures $\mathbb{E}[\hat{m}_t] = \mathbb{E}[g_t]$ and $\mathbb{E}[\hat{v}_t] = \mathbb{E}[g_t^2]$. AdamW modifies the weight update to $w_t = (1 - \eta\lambda)\,w_{t-1} - \eta\,\hat{m}_t/(\sqrt{\hat{v}_t}+\epsilon)$, applying decay $\lambda$ directly to weights rather than through the gradient.

---

In [4]:
# Contour plot with trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Beale contours
xg = np.linspace(-4.5, 4.5, 300)
yg = np.linspace(-4.5, 4.5, 300)
X, Y = np.meshgrid(xg, yg)
Z = beale(X, Y)
Z = np.log10(Z + 1)  # log scale for visibility

colors = {'SGD': 'red', 'Momentum': 'orange', 'RMSProp': 'green', 'Adam': 'blue', 'AdamW': 'purple'}

# Trajectory plot
ax = axes[0]
ax.contour(X, Y, Z, levels=30, cmap='gray', alpha=0.5)
for name, path in paths.items():
    ax.plot(path[:, 0], path[:, 1], '-', color=colors[name], label=name, alpha=0.8, lw=1.5)
ax.plot(3, 0.5, 'k*', markersize=15, label='Optimum')
ax.plot(-3, -3, 'ko', markersize=8)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Optimiser Trajectories on Beale Function')
ax.legend(fontsize=9); ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)

# Loss curves
ax = axes[1]
for name, path in paths.items():
    losses = [beale(p[0], p[1]) for p in path[::10]]  # subsample for speed
    ax.plot(np.arange(len(losses)) * 10, losses, color=colors[name], label=name)
ax.set_xlabel('Step'); ax.set_ylabel('Beale(x, y)')
ax.set_title('Loss Curves')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Optimiser Comparison: Beale Function from (-3, -3)', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/optimiser_comparison.png', dpi=100)
plt.show()

# Verify Adam-family converges closer than SGD
adam_loss = beale(*paths['Adam'][-1])
sgd_loss = beale(*paths['SGD'][-1])
assert adam_loss < sgd_loss, "Adam should outperform SGD on Beale"
print(f"\nAdam final loss ({adam_loss:.4f}) < SGD final loss ({sgd_loss:.4f}) ✓")


Adam final loss (0.0025) < SGD final loss (21.4615) ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_68517/3166745684.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Step-Size Comparison: L2 norm of parameter update at steps 100, 1000, 5000
ss_checkpoints = [100, 1000, 5000]
ss_optimisers = ['SGD', 'Momentum', 'RMSProp', 'Adam', 'AdamW']
ss_colors = {'SGD': 'red', 'Momentum': 'orange', 'RMSProp': 'green', 'Adam': 'blue', 'AdamW': 'purple'}

# Compute step sizes (L2 norm of consecutive position differences)
ss_data = {}
for name in ss_optimisers:
    p = paths[name]
    # step[i] = ||w_{i+1} - w_i||
    steps_vec = np.diff(p, axis=0)  # shape (steps, 2)
    step_norms = np.linalg.norm(steps_vec, axis=1)  # shape (steps,)
    ss_data[name] = {s: step_norms[s - 1] for s in ss_checkpoints}  # step s uses index s-1

# Grouped bar chart
ss_fig, ss_ax = plt.subplots(figsize=(9, 5))
ss_x = np.arange(len(ss_checkpoints))
ss_width = 0.15
ss_offsets = np.arange(len(ss_optimisers)) - len(ss_optimisers) / 2 + 0.5

for i, name in enumerate(ss_optimisers):
    vals = [ss_data[name][s] for s in ss_checkpoints]
    ss_ax.bar(ss_x + ss_offsets[i] * ss_width, vals, ss_width,
              label=name, color=ss_colors[name], edgecolor='black', linewidth=0.5)

ss_ax.set_xticks(ss_x)
ss_ax.set_xticklabels([f'Step {s}' for s in ss_checkpoints])
ss_ax.set_ylabel('Step Size (L2 norm of update)')
ss_ax.set_title('Effective Parameter Update Magnitude per Optimiser')
ss_ax.set_yscale('log')
ss_ax.legend(fontsize=9)
ss_ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../assets/step_size_comparison.png', dpi=100)
plt.show()

# Print numerical values
print("\nStep-size magnitudes (L2 norm of update):")
print(f"{'Optimiser':>10s}  {'Step 100':>12s}  {'Step 1000':>12s}  {'Step 5000':>12s}")
for name in ss_optimisers:
    vals = [ss_data[name][s] for s in ss_checkpoints]
    print(f"{name:>10s}  {vals[0]:12.6f}  {vals[1]:12.6f}  {vals[2]:12.6f}")

## Structured Interpretation

1. **SGD struggles** on Beale because the loss landscape has regions of very different curvature. A learning rate large enough for the flat region causes oscillation in the steep region, and vice versa.

2. **Momentum** accumulates velocity, allowing it to power through flat regions, but can overshoot in steep valleys.

3. **RMSProp** adapts per-coordinate learning rates via a running average of squared gradients. This lets it take larger steps along flat dimensions and smaller steps along steep ones.

4. **Adam** combines momentum (first moment) with RMSProp (second moment), plus bias correction. It's the de facto default for most deep learning.

5. **AdamW** decouples weight decay from the adaptive gradient update. Standard L2 regularisation in Adam is scaled by the adaptive learning rate, making its effect inconsistent. AdamW applies weight decay directly to the weights, giving more predictable regularisation. This is why AdamW is preferred for training with weight decay.

6. **For MNEMOSYNE**: We'll use AdamW as the default optimiser across all experiments, consistent with modern best practice for supervised learning tasks.